# Кластеризация и техники понижения размерности. Часть II

## Меры качества кластеризации. Практика.

### Задание 4.1
Так как изначально данные были представлены для решения задачи классификации, то они находятся в разных файлах (обучающая и тестовая выборки в соотношении 70/30). Соедините признаки так, чтобы сначала шла обучающая выборка, а затем — тестовая, и отдельно соедините значения целевых переменных (разумеется, в том же порядке).

Какая размерность получилась у набора данных с признаками?

In [2]:
import pandas as pd

# 1. Загружаем признаки
X_train = pd.read_csv('data/train.txt', sep=r'\s+', header=None)
X_test = pd.read_csv('data/test.txt', sep=r'\s+', header=None)

# 2. Объединяем признаки (сначала train, потом test)
X = pd.concat([X_train, X_test], ignore_index=True)

# 3. Загружаем целевую переменную
y_train = pd.read_csv('data/train_labels.txt', sep=r'\s+', header=None)
y_test = pd.read_csv('data/test_labels.txt', sep=r'\s+', header=None)

# 4. Объединяем метки
y = pd.concat([y_train, y_test], ignore_index=True)

# 5. Выводим размерность набора с признаками
print(f"Размерность набора данных с признаками: {X.shape}")

Размерность набора данных с признаками: (10299, 561)


### Задание 4.2
Теперь найдите число различных активностей, то есть на сколько кластеров в идеале должны разделиться наблюдения.

In [3]:
# 1. Смотрим на уникальные значения в целевой переменной
unique_labels = y[0].unique()

# 2. Выводим их количество и сами значения
print(f"Количество различных активностей (кластеров): {len(unique_labels)}")
print(f"Уникальные метки: {unique_labels}")

Количество различных активностей (кластеров): 6
Уникальные метки: [5 4 6 1 3 2]


### Задание 4.3
Далее необходимо отмасштабировать признаки. Будем использовать для этого алгоритм StandardScaler. Примените его ко всем значениям признаков и впишите в качестве ответа значение первого признака для первого объекта, предварительно округлив его до двух знаков после точки-разделителя.

In [4]:
from sklearn.preprocessing import StandardScaler

# 1. Инициализируем скейлер
scaler = StandardScaler()

# 2. Применяем масштабирование ко всем признакам
X_scaled = scaler.fit_transform(X)

# 3. Берем первое значение (первая строка, первый столбец) и округляем
first_value = X_scaled[0, 0]

print(f"Первое значение первого признака после масштабирования: {round(first_value, 2)}")

Первое значение первого признака после масштабирования: 0.21


### Задание 4.4
Пора переходить к кластеризации. Для начала определите оптимальное количество кластеров, используя внутренние меры кластеризации. Используйте все известные вам коэффициенты, реализуемые в библиотеке sklearn: коэффициент силуэта, индекс Калински — Харабаса и индекс Дэвиса — Болдина. В качестве алгоритма возьмите k-means++, в качестве значения параметра random_state — число 42.

Выведите оптимальное количество кластеров для каждой метрики, перебирая значения от 2 до 9 включительно. Также введите значение каждой метрики, округлённое до двух знаков после точки-разделителя.

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import pandas as pd

# Создаем списки для хранения результатов
results = []

K_range = range(2, 10)

for k in K_range:
    # Обучаем модель
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    # Считаем метрики
    sil_score = silhouette_score(X_scaled, labels)
    ch_score = calinski_harabasz_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    # Сохраняем в список
    results.append({
        'k': k,
        'Silhouette': sil_score,
        'Calinski-Harabasz': ch_score,
        'Davies-Bouldin': db_score
    })

# Переводим в DataFrame для красивого вывода
df_metrics = pd.DataFrame(results).set_index('k')

# Округляем для вывода
print("Значения метрик для каждого k:")
print(df_metrics.round(2))

# Находим оптимальное k для каждой метрики
best_sil_k = df_metrics['Silhouette'].idxmax()
best_ch_k = df_metrics['Calinski-Harabasz'].idxmax()
best_db_k = df_metrics['Davies-Bouldin'].idxmin()

print(f"\n--- Оптимальное количество кластеров ---")
print(f"По коэффициенту силуэта (максимум): k = {best_sil_k} (значение: {df_metrics.loc[best_sil_k, 'Silhouette']:.2f})")
print(f"По индексу Калински-Харабаса (максимум): k = {best_ch_k} (значение: {df_metrics.loc[best_ch_k, 'Calinski-Harabasz']:.2f})")
print(f"По индексу Дэвиса-Болдина (минимум): k = {best_db_k} (значение: {df_metrics.loc[best_db_k, 'Davies-Bouldin']:.2f})")

Значения метрик для каждого k:
   Silhouette  Calinski-Harabasz  Davies-Bouldin
k                                               
2        0.39            7880.81            1.07
3        0.32            5034.48            1.79
4        0.15            3696.34            2.34
5        0.13            3027.32            2.41
6        0.11            2556.54            2.38
7        0.08            2217.86            2.68
8        0.07            1975.21            2.61
9        0.07            1790.14            2.62

--- Оптимальное количество кластеров ---
По коэффициенту силуэта (максимум): k = 2 (значение: 0.39)
По индексу Калински-Харабаса (максимум): k = 2 (значение: 7880.81)
По индексу Дэвиса-Болдина (минимум): k = 2 (значение: 1.07)


### Задание 4.5
Теперь давайте оценим качество кластеризации с точки зрения внешних мер. Реализуйте кластеризацию с помощью классической версии алгоритма k-means. Пусть количество кластеров будет соответствовать количеству активностей. Задайте следующие значения параметров: random_state=42, init='random'.

В качестве ответов введите значения получившихся мер, предварительно округлив их до двух знаков после точки-разделителя.

In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import homogeneity_score, completeness_score, adjusted_rand_score

# 1. Определяем количество активностей (кластеров)
n_activities = y[0].nunique()

# 2. Обучаем K-Means с параметрами из задания
kmeans_ext = KMeans(n_clusters=n_activities, init='random', random_state=42, n_init=10)
labels_ext = kmeans_ext.fit_predict(X_scaled)

# 3. Считаем нужные метрики
homogeneity = homogeneity_score(y[0], labels_ext)
completeness = completeness_score(y[0], labels_ext)
ari = adjusted_rand_score(y[0], labels_ext)

# 4. Выводим результаты, округленные до двух знаков
print(f"Количество кластеров: {n_activities}")
print(f"Однородность (Homogeneity): {round(homogeneity, 2)}")
print(f"Полнота (Completeness): {round(completeness, 2)}")
print(f"Adjusted Rand Index (ARI): {round(ari, 2)}")

Количество кластеров: 6
Однородность (Homogeneity): 0.54
Полнота (Completeness): 0.58
Adjusted Rand Index (ARI): 0.42


### Задание 4.6
Выясните, к каким кластерам были преимущественно отнесены различные активности (т. е. в какой кластер попало большинство наблюдений с этой активностью). Нумерацию кластеров начинайте с 1.

In [7]:
import pandas as pd

# 1. Создаем кросс-таблицу (матрицу сопряженности)
# Индекс — это истинные активности (от 1 до 6)
# Столбцы — это предсказанные кластеры (0, 1, 2...)
# Прибавляем 1 к меткам, чтобы кластеры нумеровались с 1, как просят в задании
crosstab = pd.crosstab(
    index=y[0], 
    columns=labels_ext + 1
)

# 2. Выводим таблицу
print("Матрица сопряженности (строки - истинные активности, столбцы - кластеры):")
display(crosstab)

# 3. Находим доминирующий кластер для каждой активности
print("\n--- Преимущественный кластер для каждой активности ---")
for activity in crosstab.index:
    # Находим столбец (кластер) с максимальным значением в строке
    dominant_cluster = crosstab.loc[activity].idxmax()
    count = crosstab.loc[activity].max()
    print(f"Активность {activity}: преимущественно в кластере {dominant_cluster} (попаданий: {count})")
    

Матрица сопряженности (строки - истинные активности, столбцы - кластеры):


col_0,1,2,3,4,5,6
0,,,,,,
1,741,897,0,0,84,0
2,297,1236,2,0,9,0
3,882,310,0,0,214,0
4,0,1,447,91,0,1238
5,0,0,560,0,0,1346
6,0,5,329,1556,0,54



--- Преимущественный кластер для каждой активности ---
Активность 1: преимущественно в кластере 2 (попаданий: 897)
Активность 2: преимущественно в кластере 2 (попаданий: 1236)
Активность 3: преимущественно в кластере 1 (попаданий: 882)
Активность 4: преимущественно в кластере 6 (попаданий: 1238)
Активность 5: преимущественно в кластере 6 (попаданий: 1346)
Активность 6: преимущественно в кластере 4 (попаданий: 1556)


### Задание 4.7
Теперь попробуйте реализовать алгоритм k-means для двух кластеров (для того числа активностей, которое является оптимальным с точки зрения внутренних мер) и снова посмотреть, как алгоритм разобьёт активности по кластерам. Как и в предыдущем задании, нумерацию кластеров начинайте с 1.

In [8]:
from sklearn.cluster import KMeans
import pandas as pd

# 1. Обучаем K-Means для 2 кластеров
kmeans_2 = KMeans(n_clusters=2, init='random', random_state=42, n_init=10)
labels_2 = kmeans_2.fit_predict(X_scaled)

# 2. Строим кросс-таблицу (прибавляем 1 к меткам)
crosstab_2 = pd.crosstab(
    index=y[0], 
    columns=labels_2 + 1
)

print("Матрица сопряженности для 2 кластеров:")
display(crosstab_2)

# 3. Выводим доминирующий кластер для каждой активности
print("\n--- Преимущественный кластер для каждой активности ---")
for activity in crosstab_2.index:
    dominant_cluster = crosstab_2.loc[activity].idxmax()
    print(f"Активность {activity}: кластер {dominant_cluster}")

Матрица сопряженности для 2 кластеров:


col_0,1,2
0,,
1,1722,0
2,1536,8
3,1406,0
4,3,1774
5,0,1906
6,12,1932



--- Преимущественный кластер для каждой активности ---
Активность 1: кластер 1
Активность 2: кластер 1
Активность 3: кластер 1
Активность 4: кластер 2
Активность 5: кластер 2
Активность 6: кластер 2


### Задание 4.8
Вычислите значение полноты для разбиения на два кластера алгоритмом k-means. Ответ округлите до двух знаков после точки-разделителя.

In [9]:
from sklearn.metrics import completeness_score

# Считаем полноту (Completeness) для 2 кластеров
completeness_2 = completeness_score(y[0], labels_2)

print(f"Полнота (Completeness) для 2 кластеров: {round(completeness_2, 2)}")

Полнота (Completeness) для 2 кластеров: 0.98


### Задание 4.10
Давайте сравним полученный результат с агломеративной иерархической кластеризацией. Реализуйте её также для двух кластеров и вычислите значение полноты.

In [10]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import completeness_score

# 1. Обучаем агломеративную кластеризацию для 2 кластеров
agg_model_2 = AgglomerativeClustering(n_clusters=2)
labels_agg_2 = agg_model_2.fit_predict(X_scaled)

# 2. Считаем полноту (Completeness)
completeness_agg_2 = completeness_score(y[0], labels_agg_2)

print(f"Полнота (Completeness) для агломеративной кластеризации (2 кластера): {round(completeness_agg_2, 2)}")

Полнота (Completeness) для агломеративной кластеризации (2 кластера): 1.0
